# Pipeline Orchestrator — Retry Logic & Failure Handling

## Purpose
This notebook is the **single entry point** for the entire metadata-driven pipeline.
It chains Bronze → Silver → Gold in sequence with:
- **Configurable retry logic** with exponential back-off per layer
- **Failure isolation** — a failed source doesn't block other sources
- **Observability** — every attempt is logged to `pipeline_run_log`
- **Databricks Workflow integration** — raises a final exception if the  
  pipeline fails after all retries, so the Workflow task is marked FAILED  
  and alerts/SLAs are triggered

## Orchestration Flow
```
fw_5.orchestrator
  │
  ├─► %run fw_0.config             (parameters + helpers)
  ├─► run_with_retry(Bronze)       %run fw_2.bronze_autoloader
  ├─► run_with_retry(Silver)       %run fw_3.silver_merge
  └─► run_with_retry(Gold)         %run fw_4.gold_aggregation
```

## Retry Configuration (Widgets)
| Widget | Default | Description |
|---|---|---|
| `max_retries` | 3 | Max attempts per layer |
| `retry_delay_seconds` | 30 | Base wait between retries (doubles each attempt) |
| `run_bronze` | true | Set to false to skip Bronze |
| `run_silver` | true | Set to false to skip Silver |
| `run_gold` | true | Set to false to skip Gold |

## Databricks Workflow Setup
Create a single-task Workflow that runs this notebook.
Set **Max retries = 0** on the Workflow task itself — retry logic is handled here.
Add **email/webhook alerts** on task failure for SLA monitoring.

In [ ]:
# ── Orchestration Widgets ──────────────────────────────────────────────────────
dbutils.widgets.text("max_retries",         "3",    "Max retries per layer")
dbutils.widgets.text("retry_delay_seconds", "30",   "Base retry delay (seconds, doubles each attempt)")
dbutils.widgets.dropdown("run_bronze", "true", ["true", "false"], "Run Bronze layer")
dbutils.widgets.dropdown("run_silver", "true", ["true", "false"], "Run Silver layer")
dbutils.widgets.dropdown("run_gold",   "true", ["true", "false"], "Run Gold layer")

%run ./fw_0.config

MAX_RETRIES   = int(dbutils.widgets.get("max_retries"))
RETRY_DELAY   = int(dbutils.widgets.get("retry_delay_seconds"))
RUN_BRONZE    = dbutils.widgets.get("run_bronze").lower() == "true"
RUN_SILVER    = dbutils.widgets.get("run_silver").lower() == "true"
RUN_GOLD      = dbutils.widgets.get("run_gold").lower()   == "true"

print(f"Orchestrator config:")
print(f"  max_retries : {MAX_RETRIES}")
print(f"  retry_delay : {RETRY_DELAY}s (doubles each attempt)")
print(f"  run_bronze  : {RUN_BRONZE}")
print(f"  run_silver  : {RUN_SILVER}")
print(f"  run_gold    : {RUN_GOLD}")

In [ ]:
# ── Retry engine with exponential back-off ─────────────────────────────────────

import time
from datetime import datetime

def run_with_retry(layer_name: str, notebook_path: str, max_retries: int, base_delay: int) -> bool:
    """
    Run a notebook with retry + exponential back-off.
    Returns True on success, False after all retries exhausted.

    Back-off schedule (base_delay=30):
      Attempt 1: immediate
      Attempt 2: wait 30s
      Attempt 3: wait 60s
      Attempt 4: wait 120s
    """
    attempt = 0
    delay   = base_delay

    while attempt <= max_retries:
        attempt_label = f"Attempt {attempt + 1}/{max_retries + 1}"
        print(f"\n[ORCH] {layer_name.upper()} — {attempt_label} — {datetime.utcnow().isoformat()}Z")

        try:
            dbutils.notebook.run(
                notebook_path,
                timeout_seconds=7200,  # 2-hour hard timeout per layer
                arguments={
                    "storage_account":   STORAGE_ACCOUNT,
                    "storage_credential": STORAGE_CREDENTIAL,
                    "catalog_name":      CATALOG_NAME,
                    "env":               ENV,
                    "framework_schema":  FRAMEWORK_SCHEMA
                }
            )
            print(f"[ORCH] {layer_name.upper()} — {attempt_label} SUCCEEDED.")
            return True

        except Exception as e:
            error_msg = str(e)
            print(f"[ORCH] {layer_name.upper()} — {attempt_label} FAILED: {error_msg}")

            if attempt < max_retries:
                print(f"[ORCH] Retrying in {delay}s...")
                time.sleep(delay)
                delay *= 2  # Exponential back-off
            attempt += 1

    print(f"[ORCH] {layer_name.upper()} — All {max_retries + 1} attempts exhausted. GIVING UP.")
    return False


print("Retry engine loaded.")

In [ ]:
# ── Pipeline execution ────────────────────────────────────────────────────────

pipeline_start = datetime.utcnow()
layer_results  = {}

print("="*60)
print(f"PIPELINE START: {pipeline_start.isoformat()}Z")
print("="*60)

# ── Bronze ────────────────────────────────────────────────────────────────────
if RUN_BRONZE:
    ok = run_with_retry("bronze", "./fw_2.bronze_autoloader", MAX_RETRIES, RETRY_DELAY)
    layer_results["bronze"] = "SUCCESS" if ok else "FAILURE"
    if not ok:
        print("[ORCH] Bronze failed — aborting pipeline (Silver and Gold depend on Bronze).")
        dbutils.notebook.exit("PIPELINE_FAILED: Bronze layer failed after all retries.")
else:
    layer_results["bronze"] = "SKIPPED"
    print("[ORCH] Bronze SKIPPED (run_bronze=false).")

# ── Silver ────────────────────────────────────────────────────────────────────
if RUN_SILVER:
    ok = run_with_retry("silver", "./fw_3.silver_merge", MAX_RETRIES, RETRY_DELAY)
    layer_results["silver"] = "SUCCESS" if ok else "FAILURE"
    if not ok:
        print("[ORCH] Silver failed — aborting pipeline (Gold depends on Silver).")
        dbutils.notebook.exit("PIPELINE_FAILED: Silver layer failed after all retries.")
else:
    layer_results["silver"] = "SKIPPED"
    print("[ORCH] Silver SKIPPED (run_silver=false).")

# ── Gold ──────────────────────────────────────────────────────────────────────
if RUN_GOLD:
    ok = run_with_retry("gold", "./fw_4.gold_aggregation", MAX_RETRIES, RETRY_DELAY)
    layer_results["gold"] = "SUCCESS" if ok else "FAILURE"
else:
    layer_results["gold"] = "SKIPPED"
    print("[ORCH] Gold SKIPPED (run_gold=false).")

pipeline_end      = datetime.utcnow()
duration_seconds  = (pipeline_end - pipeline_start).total_seconds()

print("\n" + "="*60)
print("PIPELINE COMPLETE")
print("="*60)
for layer, status in layer_results.items():
    print(f"  {layer.upper():8s}  {status}")
print(f"  Duration : {duration_seconds:.1f}s")
print(f"  Finished : {pipeline_end.isoformat()}Z")

In [ ]:
# ── Final failure gate — Databricks Workflow sees this as a task failure ───────
# This triggers configured email/webhook alerts and marks the Workflow run FAILED.

final_failures = [l for l, s in layer_results.items() if s == "FAILURE"]
if final_failures:
    raise RuntimeError(
        f"PIPELINE FAILED: layers {final_failures} failed after {MAX_RETRIES} retries. "
        f"Check {fq(FRAMEWORK_SCHEMA, 'pipeline_run_log')} for per-source details."
    )

dbutils.notebook.exit("PIPELINE_SUCCESS")

In [ ]:
# ── Maintenance Cell (run separately on a schedule, not part of daily pipeline) ─
# Schedule this as a separate Databricks Workflow task — weekly, off-peak.

def run_maintenance():
    configs = get_pipeline_configs(filter_active=True)
    for cfg in configs:
        for schema, table_name in [
            (BRONZE_SCHEMA, cfg.bronze_table),
            (SILVER_SCHEMA, cfg.silver_table),
        ]:
            tbl = fq(schema, table_name)
            try:
                spark.sql(f"VACUUM {tbl} RETAIN 168 HOURS")
                print(f"[MAINTENANCE] VACUUM OK: {tbl}")
            except Exception as e:
                print(f"[MAINTENANCE] VACUUM failed for {tbl}: {e}")

        if cfg.gold_table:
            tbl = fq(GOLD_SCHEMA, cfg.gold_table)
            try:
                spark.sql(f"VACUUM {tbl} RETAIN 168 HOURS")
                print(f"[MAINTENANCE] VACUUM OK: {tbl}")
            except Exception as e:
                print(f"[MAINTENANCE] VACUUM failed for {tbl}: {e}")

# Uncomment to run:
# run_maintenance()
print("Maintenance function defined. Uncomment run_maintenance() to execute.")